# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [1]:
! pip install -q schedule pytest

In [2]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import time
import requests
import schedule
import re
from bs4 import BeautifulSoup

## Задание 1. Сбор данных об одной книге (20 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [3]:
# import time
# import requests
# import schedule
# from bs4 import BeautifulSoup 
# оказывается импорты и фукцнции можно тянуть из пред ячеек юпитер ноутбука, прикольно :)

def get_book_data(url: str, timeout: int = 15, debug: bool = False) -> dict:
    """
    Функция загружает страницу одной книги по заданному url и возращает результаты парсинга страницы с книгой.

    Args:
        url (str): полный url детальной страницы книги.
        timeout (int): таймаут запроса в секундах (по умолчанию 15).

    Returns:
        dict: Словарь с ключами:
            - title (str | None): Название книги
            - price (float | None): Значение цены
            - rating (int): Рейтинг по звездам (1..5; 0 - если не найден).
            - availability (int): Количество в наличии (целое число; 0 - есои не найден).
            - description (str | None): Краткое описание.
            - product_information (dict): Пары из таблицы Product Information.
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ

    response = requests.get(url, timeout=timeout)  
    response.raise_for_status()  # исключение, если код ответа != 200

    # без utf8 символ £ спарсится как Â£
    response.encoding = "utf-8"

    soup = BeautifulSoup(response.text, "html.parser")  # , from_encoding="utf-8"
    
    main = soup.select_one("div.product_main")  # главный div

    #  1. название
    title_tag = main.find("h1") if main else None  
    title = title_tag.get_text(strip=True) if title_tag else None
    if debug:
        print("название: ", title)

    
    #  2. цена
    price_tag = main.select_one(".price_color") if main else None  
    price_raw = price_tag.get_text(strip=True) if price_tag else None  
    price = None
    
    if price_raw:
        m = re.search(r"(\d+(?:\.\d+)?)", price_raw)  # забираем число с точкой, без сивмола валюты
        price = float(m.group(1)) if m else None

        if debug:
            print("цена строкой: ", price_raw)
            print("цена числом: ", price)

    
    #  3. рейтинг
    rating_words_to_int = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}  # "No rating": 0,
    rating = 0  # дефолтный
    
    rating_tag = main.select_one(".star-rating") if main else None  

    if rating_tag:  
        for cls in rating_tag.get("class", []):  
            if cls in rating_words_to_int:  
                rating = rating_words_to_int[cls]
                if debug:
                    print("рейтинг: ", rating)
                break

    #  4. наличие: 
    #  <p class="instock availability"><i class="icon-ok"></i>In stock (22 available)</p>
    
    availability = 0  
    availability_tag = soup.select_one("p.instock.availability")  
    
    if availability_tag:  
        availability_text = availability_tag.get_text(strip=True)  
        
        match = re.search(r"(\d+)", availability_text)  # ищем число
        
        if match:  
            availability = int(match.group(1))
            if debug:
                print("строка с инф о наличии: ", availability_text)
                print("наличие в штуках: ", availability)


    #  5. Описание книги 
    #  <div id="product_description" class="sub-header"><h2></h2></div><p>
    
    description = None
    desc_header = soup.find(id="product_description")
    if desc_header:
        desc_paragraph = desc_header.find_next("p")
        if desc_paragraph:
            description = desc_paragraph.get_text(strip=True)
            if debug:
                print(f"\nОписание книги: {description} \n")
    
    
    #  6. Таблица product information "table table-striped"
    product_info_table = soup.select_one("table.table.table-striped")
    product_information = {} 
    if product_info_table:  
        rows = product_info_table.select("tr")  

        if debug:
            print("\nпробуем получить параметры книги:\n")
            
        for row in rows:  
            th = row.find("th")  
            td = row.find("td")  
            if th and td:  
                key = th.get_text(strip=True)  
                value = td.get_text(strip=True)  
                product_information[key] = value  
                if debug:
                    print(f"Параметр {key} = {value}")
                
    #  7. итого

    #  price_raw и availability_text не возвращается в результатах, но они есть в режиме отладки
    result = {
        "title": title,  
        "price": price,  
        "rating": rating,  
        "availability": availability,  
        "description": description,  
        "product_information": product_information,  
    }

    return result

print("\nИспользуем debug=True для демонстрации / отладки работы функции: \n")
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
res = get_book_data(book_url, debug=True)
# print(res)

    # КОНЕЦ ВАШЕГО РЕШЕНИЯ


Используем debug=True для демонстрации / отладки работы функции: 

название:  A Light in the Attic
цена строкой:  £51.77
цена числом:  51.77
рейтинг:  3
строка с инф о наличии:  In stock (22 available)
наличие в штуках:  22

Описание книги: It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silv

In [4]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'title': 'A Light in the Attic',
 'price': 51.77,
 'rating': 3,
 'availability': 22,
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?And who put you up

## Задание 2. Сбор данных обо всех книгах (20 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [ ]:
def scrape_books(base_pattern: str,
                 save_to_file: bool = False,
                 timeout: int = 15,
                 debug: bool = False) -> list:
    """
    
    Функция обходит каталог книг по шаблону url формата '.../page-{N}.html' и
    собирает данные всех книг со всех страниц, использую кнопку Next для определения,
    есть ли еще доступные страницы листингов.

    Args:
        - base_pattern (str): шаблон url с подстановкой {N}. 
        Пример: 'http://books.toscrape.com/catalogue/page-{N}.html'.
        
        - save_to_file (bool): если True — сохраняет результат в books_data.txt (по 1-й строке на книгу).
        
        - timeout (int): Таймаут HTTP-запроса в секундах
        - debug (bool): для режима отладки

    Returns:
        list[dict]: Список словарей, каждый из них - результат вызова get_book_data() для одной книги.
        
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    all_books = []
    
    n = 0

    while True:
        
        n += 1
        
        page_url = base_pattern.replace("{N}", str(n))
        
        if debug:
            print(f"\nПарсим страницу №{n}: {page_url}")
            
        resp = requests.get(page_url, timeout=timeout)

        if resp.status_code == 404:
            if debug:
                print("err 404, что-то пошло не так :)")
            break

        if resp.status_code != 200:  # обрабатываем только 200 ответы, без 304, 301 
            resp.raise_for_status()

        resp.encoding = "utf-8"
        soup = BeautifulSoup(resp.text, "html.parser")

        cards = soup.select("article.product_pod")  
        
        if debug:
            print("карточек на странице: ", len(cards))

        #  кнопки "Next" для перехода на след страницу имеют ссылки вида: page-4.html, соотв. нужнен base dir
        #  чтобы сформировать финальный url
        
        cut = page_url.rfind("/")                                  
        base_dir = page_url[:cut + 1] if cut != -1 else page_url + "/"

        SITE_ROOT = "https://books.toscrape.com/"

        for card in cards:
            a_tag = card.select_one("h3 a")
            if not a_tag:
                continue
            href = a_tag.get("href", "")
            
            #  делаем поддержку всех типов ссылок: абсолютные, короткие "//somelink", относительные, и ссылки требующих base_dir
            if href.startswith("http://") or href.startswith("https://"):
                book_url = href
            elif href.startswith("/"):
                book_url = SITE_ROOT + href.lstrip("/")
            else:
                cut = page_url.rfind("/")
                base_dir = page_url[:cut + 1] if cut != -1 else page_url + "/"
                book_url = base_dir + href

            if debug:
                #  print("raw ссылка: ", href)
                print("ссылка на книгу: ", book_url)

            try:
                data = get_book_data(book_url, timeout=timeout, debug=False)
                all_books.append(data)
            except Exception as err:
                if debug:
                    print("ошибка парсинга: ", err)
                    
        #  ищем ссылку на след. страницу (кнопка Next), если она есть - продолжаем while, иначе останавливаем цикл
        has_next = soup.select_one("li.next > a") is not None 
        if not has_next:
            if debug:
                print("нет ссылки на слет страницу ")
            break  # выходим из while(true)
            
        if n > 3: # чтобы перепарсинг весь не делать
            break

    if save_to_file:
        if debug:
            print("\сохранение в файл")
        with open("books_data.txt", "w", encoding="utf-8") as f:
            for item in all_books:
                f.write(str(item) + "\n")

    return all_books

    
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

# денострация работы в режиме отладки 
# в итоге отключил режим отладки, т.к. огромное полотно получается print'ов
# pattern = "http://books.toscrape.com/catalogue/page-{N}.html"
# result = scrape_books(pattern, save_to_file=True, debug=False)



In [ ]:
# Проверка работоспособности функции

# LOOOK!!! я парсил еще за неделю до сдачи, в день сдачи доделываю все в полсдений момент (я же мозг :)
# и чтобы все спарсилось после перезапуская ядра (а тут что-то около 25 страниц по 20 книг) нужно примерно 30 минут (в прошлый раз так было)
# а без этого ячейка не покажет рузльтаты. я сделаю ограничение для демо на 3 страниц в коде (epic, да)


pattern = "http://books.toscrape.com/catalogue/page-{N}.html"
res = scrape_books(pattern, save_to_file=True)
print(type(res), len(res))


## Задание 3. Настройка регулярной выгрузки (10 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [ ]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ

import time
import schedule

def job(demo_mode: bool = False):
    print(f"{time.strftime('%H:%M:%S')} старт задачи")
    pattern = "http://books.toscrape.com/catalogue/page-{N}.html"
    
    if not demo_mode:
        scrape_books(pattern, save_to_file=True, debug=False)
    else:
        print("ячейка запущена в демо режиме, без реального парсинга, для демонстрации работы планировщика")
        
    print(f"{time.strftime('%H:%M:%S')} готово")

schedule.every().day.at("23:41").do(job, demo_mode=True) # "14:15" "23:20" "19:00"

print("планировщик запущен, ожидаем запуск")
while True:
    schedule.run_pending()
    time.sleep(3)   
    
# КОНЕЦ ВАШЕГО РЕШЕНИЯ

планировщик запущен, ожидаем запуск
23:41:01 старт задачи
ячейка запущена в демо режиме, без реального парсинга, для демонстрации работы планировщика
23:41:01 готово


## Задание 4. Написание автотестов (15 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [5]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
! pytest ./tests/test_scraper.py

============================= test session starts =============================
platform win32 -- Python 3.13.5, pytest-8.3.4, pluggy-1.5.0
rootdir: C:\Users\0x39F963\Documents\! МФТИ - УЧЕБА 1 СЕМЕСТР\Python
plugins: anyio-4.7.0
collected 3 items

tests\test_scraper.py ...                                                [100%]

============================= 3 passed in 17.39s ==============================


## Задание 5. Оформление проекта на GitHub и работа с Git (35 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds_2025.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

In [ ]:
# РЕПОЗИТОРИЙ:   https://github.com/0x39f963/novikov_dz3